In [6]:
!pip install gradio

In [8]:
import gradio as ui
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC


def train_and_compare(
    models_to_run, lr_c, rf_estimators, rf_depth, svm_c, svm_kernel
):
    if not models_to_run:
        return "Ошибка: Выберите хотя бы одну модель для обучения!", None

    # 1. Генерация данных (исправлен аргумент test_size)
    X, y = make_classification(
        n_samples=400, n_features=10, n_informative=5, random_state=42
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    results = []

    # 2. Инициализация моделей с выбранными из интерфейса параметрами
    all_models = {
        "Логистическая регрессия": LogisticRegression(C=lr_c, max_iter=1000),
        "Случайный лес": RandomForestClassifier(
            n_estimators=int(rf_estimators),
            max_depth=int(rf_depth) if rf_depth > 0 else None,
            random_state=42,
        ),
        "Метод опорных векторов (SVM)": SVC(C=svm_c, kernel=svm_kernel),
    }

    # 3. Обучение выбранных моделей
    for name in models_to_run:
        model = all_models[name]
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        # Расчет метрик
        acc = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds)
        rec = recall_score(y_test, preds)

        results.append(
            {"Модель": name, "Accuracy": acc, "Precision": prec, "Recall": rec}
        )

    df_res = pd.DataFrame(results)

    # 4. Построение графиков
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(df_res["Модель"]))
    width = 0.25

    ax.bar(x - width, df_res["Accuracy"], width, label="Accuracy", color="#4f46e5")
    ax.bar(x, df_res["Precision"], width, label="Precision", color="#06b6d4")
    ax.bar(x + width, df_res["Recall"], width, label="Recall", color="#10b981")

    ax.set_ylabel("Значение метрики")
    ax.set_title("Сравнение моделей с индивидуальными параметрами")
    ax.set_xticks(x)
    ax.set_xticklabels(df_res["Модель"], rotation=15)
    ax.set_ylim(0, 1.1)
    ax.legend(loc="lower right")
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()

    # Текстовый вывод
    text_output = df_res.to_string(index=False)

    return text_output, fig


# Построение интерфейса Gradio
with ui.Blocks(title="ML Аналитика") as demo:
    ui.Markdown("# 📊 Анализ данных и кастомизация ML-моделей")

    with ui.Row():
        # ЛЕВАЯ КОЛОНКА: Настройки моделей и выбор параметров
        with ui.Column(scale=1):
            ui.Markdown("### 1. Настройка параметров")

            with ui.Tab("Логистическая регрессия"):
                lr_c = ui.Slider(
                    minimum=0.01,
                    maximum=10.0,
                    value=1.0,
                    step=0.1,
                    label="Регуляризация (C)",
                )

            with ui.Tab("Случайный лес"):
                rf_estimators = ui.Slider(
                    minimum=10,
                    maximum=200,
                    value=100,
                    step=10,
                    label="Количество деревьев",
                )
                rf_depth = ui.Slider(
                    minimum=0,
                    maximum=20,
                    value=5,
                    step=1,
                    label="Макс. глубина (0 — без ограничений)",
                )

            with ui.Tab("SVM"):
                svm_c = ui.Slider(
                    minimum=0.1,
                    maximum=10.0,
                    value=1.0,
                    step=0.5,
                    label="Штраф за ошибку (C)",
                )
                svm_kernel = ui.Dropdown(
                    choices=["linear", "rbf", "poly"],
                    value="rbf",
                    label="Ядро (Kernel)",
                )

            ui.Markdown("### 2. Запуск")
            model_chooser = ui.CheckboxGroup(
                choices=[
                    "Логистическая регрессия",
                    "Случайный лес",
                    "Метод опорных векторов (SVM)",
                ],
                label="Какие модели обучить?",
                value=["Логистическая регрессия", "Случайный лес"],
            )

            btn_run = ui.Button("🚀 Запустить сравнение", variant="primary")

        # ПРАВАЯ КОЛОНКА: Графики и таблицы результатов
        with ui.Column(scale=1.5):
            ui.Markdown("### 3. Результаты обучения")
            text_results = ui.Textbox(
                label="Таблица метрик качества", max_lines=10, interactive=False
            )
            plot_results = ui.Plot(label="Диаграмма сравнения")

    # Передача ВСЕХ настроек интерфейса в функцию обучения
    btn_run.click(
        fn=train_and_compare,
        inputs=[
            model_chooser,
            lr_c,
            rf_estimators,
            rf_depth,
            svm_c,
            svm_kernel,
        ],
        outputs=[text_results, plot_results],
    )

# Запуск приложения
demo.launch(inline=True, share=False)


C:\Users\masha\anaconda3\Lib\site-packages\gradio\layouts\column.py:59: UserWarning: 'scale' value should be an integer. Using 1.5 will cause issues.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
